# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`
This notebook demonstrates how to load and programmatically explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library, following best practices for referencing data elements by their `@id` values.

### Dataset Source
The dataset is described and structured following the Croissant schema standard and is publicly available via a schema URL.

In [ ]:
# Ensure mlcroissant is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and basic structure using `mlcroissant`. The Croissant schema URL is specified below.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display a summary
md = dataset.metadata
print(f"Name: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Identifier: {md.identifier}")
print(f"License: {md.license}")
print(f"Keywords: {md.keywords if hasattr(md,'keywords') else ''}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities (`RecordSets`, `Fields`, `Columns`) are referenced by their `@id` according to the Croissant schema. This enables programmatic and reliable access.

In [ ]:
# List all RecordSets and their fields by @id
record_sets = dataset.record_sets  # This is a list of mlcroissant.RecordSet objects
print(f"Found {len(record_sets)} Record Sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    field_ids = [f.id for f in rs.fields]
    print(f"    Field @ids: {field_ids}")

Let's examine the first `RecordSet` and inspect a few records.

In [ ]:
# Preview records from the first RecordSet
if record_sets:
    main_record_set = record_sets[0]
    main_record_set_id = main_record_set.id
    print(f"\nExamining first RecordSet: {main_record_set_id}")
    sample_records = list(dataset.records(record_set=main_record_set_id))
    for i, rec in enumerate(sample_records[:3]):
        print(f"Record {i+1}:")
        for k, v in rec.items():
            print(f"  {k}: {v}")
        print("---")

## 3. Data Extraction
Load data from the available record sets into pandas DataFrames for further analysis. All references are made using the appropriate `@id` values.

In [ ]:
# Extract all record set data as DataFrames
import collections

dataframes = collections.OrderedDict()
for rs in record_sets:
    rs_id = rs.id
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {list(df.columns)}")
    print(f"  Shape: {df.shape}\n")

We will focus analysis on the first record set, as it's typically the main data table.

Let's examine the top records and column IDs.

In [ ]:
# Select main dataframe
main_df = list(dataframes.values())[0]
main_rs_id = list(dataframes.keys())[0]
print(f"Columns (@ids): {list(main_df.columns)}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Select a numeric field (by its `@id`) for basic analysis. We'll perform filtering, normalization, and group-by aggregation.

Replace the variable `numeric_field_id` below with an `@id` from the column list above, such as a measurement value or count.

In [ ]:
# Choose a numeric field for analysis (replace with actual @id from your data when known)
possible_numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
print("Auto-detected numeric fields:", possible_numeric_fields)

# For demonstration, try 'cr:peptide_count' if it exists else pick one
if 'cr:peptide_count' in main_df.columns:
    numeric_field_id = 'cr:peptide_count'
elif possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    raise ValueError("No numeric fields available in RecordSet.")

threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
display(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id}:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if available
candidate_cat_fields = [col for col in main_df.columns if main_df[col].dtype == 'object']
if candidate_cat_fields:
    group_field_id = candidate_cat_fields[0]
    print(f"Grouping results by '{group_field_id}' (first categorical field)...")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped_df.head())
else:
    print("No categorical fields available for grouping.")

## 5. Visualization
Visualize the filtered and transformed data using common plotting tools such as matplotlib or seaborn.

We'll plot the distribution of the normalized numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Frequency")
plt.show()

# Optionally, if a group_field exists, plot means
if 'group_field_id' in locals():
    plt.figure(figsize=(10,5))
    grouped_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    grouped_means.plot.bar()
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we:
- Demonstrated loading a Croissant-structured FAIR² dataset using only open-source Python tools (`mlcroissant`, `pandas`, `matplotlib`, `seaborn`).
- Listed the available record sets and their field `@id`s.
- Extracted records and performed example filtering, normalization, and grouping using these `@id`s.
- Created visualizations of numeric fields to enable fast insights.

For further data science analyses and reproducible workflows, always reference dataset entities by their explicit Croissant schema `@id` fields.